In [1]:

import pandas as pd
import os 

# INPUTS

root_directory = '../../50 KM Group/Royalties/Statements/Karen/Netease/'
outputdirectory = '../../50 KM Group/Royalties/Statements/Karen/_output/'

#contract = 'Main contract' 
contract = 'Contract 5 songs'
statement = '2025 Q3'
month1 = '2025-07'
month2 = '2025-08'
month3 = '2025-09'

# end inputs

subdirectory = '/as sent by Netease/'

directory = root_directory + contract + '/' + statement + subdirectory 

outputfilename = f"{outputdirectory}{contract}_{statement}.xlsx"

def find_xls_files(directory):
    xls_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if '结算报表' in file:
                xls_files.append(os.path.join(root, file))
    return xls_files 

def read_each_sheet(file,m1,m2,m3):
    excel_data = pd.ExcelFile(file)
    print(f"\nreading {file}, all sheets")
    data_frames = []
    for sheet_name in excel_data.sheet_names:
        df = excel_data.parse(sheet_name)
        print(f"Reading sheet: {sheet_name}, Rows: {df.shape[0]}, Columns: {df.shape[1]}")
        if m1 in sheet_name:
            df['月份']= m1
        elif m2 in sheet_name:
            df['月份']= m2
        elif m3 in sheet_name:
            df['月份']= m3
        if "免费" in sheet_name:
            df['Total streams and download']=df['总播放量']+df['总下载量']
            df['类型']='免费'
            df.rename(columns={'本月分成收益费用': '本月实际分成收益费用 - royalties'}, inplace=True)
            data_frames.append(df)
        elif '付费2' in sheet_name:
            df['Total streams and download']= df['总播放量']+df['总下载量'] 
            df['类型']='付费'
            df.rename(columns={'本月实际分成收益费用': '本月实际分成收益费用 - royalties'}, inplace=True)
            data_frames.append(df)
        elif '订阅' in sheet_name:
            df['Total streams and download']= df['总播放量']+df['总下载量'] 
            df['类型']='付费'
            df.rename(columns={'本月实际分成收益费用': '本月实际分成收益费用 - royalties'}, inplace=True)
            data_frames.append(df)
        elif '付费单曲' in sheet_name:
            df.rename(columns={'销售数量': 'Total streams and download'}, inplace=True)
            df['类型']='付费单曲'
            df.rename(columns={'本月实际销售收益费用': '本月实际分成收益费用 - royalties'}, inplace=True)
            data_frames.append(df)
        elif 'K歌' in sheet_name:
            df['类型']='K歌'
            df['Total streams and download']=0
            df.rename(columns={'本月单价收益费用': '本月实际分成收益费用 - royalties'}, inplace=True)
            data_frames.append(df)
    df = pd.concat(data_frames, ignore_index=True)
    df = df[df['ISRC'] != '合计']
    df['月份'] = df['月份'].str.replace('-', '')
    print(f"\nThe combined DataFrame has {df.shape[0]} rows and {len(df.columns)} columns.")
    print(f"Royalty: {df['本月实际分成收益费用 - royalties'].sum()}. Units: {df['Total streams and download'].sum()}")
    return df


listoffiles = find_xls_files(directory)
for file in listoffiles:
    df = read_each_sheet(file, month1, month2, month3)
df['Statement Quarter']=statement
df['Contract']=contract

df["Sales_date"] = pd.to_datetime(df["月份"], format="%Y%m", errors="coerce")
df["Sales Month"] = df["Sales_date"].dt.strftime("%Y %m")
df["Sales Quarter"] = (
    df["Sales_date"].dt.to_period("Q").astype(str).str.replace("Q", " Q")
)

df =df.drop(columns=['Sales_date'])

df.to_excel(outputfilename, engine='openpyxl', index=True)



reading ../../50 KM Group/Royalties/Statements/Karen/Netease/Contract 5 songs/2025 Q3/as sent by Netease/431A202404002900-NCM-结算报表-北京兴荷娱乐有限公司(2025.07-2025.09).xlsx, all sheets
Reading sheet: 总计表, Rows: 23, Columns: 5
Reading sheet: 免费2025-07, Rows: 6, Columns: 18
Reading sheet: 免费2025-08, Rows: 6, Columns: 18
Reading sheet: 免费2025-09, Rows: 6, Columns: 18
Reading sheet: 订阅2025-07, Rows: 6, Columns: 13
Reading sheet: 订阅2025-08, Rows: 6, Columns: 13
Reading sheet: 订阅2025-09, Rows: 6, Columns: 13
Reading sheet: 付费单曲2025-07, Rows: 3, Columns: 14
Reading sheet: 付费单曲2025-08, Rows: 3, Columns: 14
Reading sheet: 付费单曲2025-09, Rows: 5, Columns: 14

The combined DataFrame has 38 rows and 23 columns.
Royalty: 53443.28222763618. Units: 16785409
